In [ ]:
!pip install -q sentence-transformers pandas tqdm

In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

Saving guide_extracted_skills.csv to guide_extracted_skills.csv
Saving video_extracted_skills.csv to video_extracted_skills.csv


In [ ]:
video_df = pd.read_csv("/content/video_extracted_skills.csv")
guide_df = pd.read_csv("/content/guide_extracted_skills.csv")

In [ ]:
import re

arabic_number_words = {
    "واحد": "1",
    "واحدة": "1",
    "اثنان": "2",
    "اثنين": "2",
    "إثنان": "2",
    "إثنين": "2",
    "ثلاثة": "3",
    "ثلاث": "3",
    "أربعة": "4",
    "اربعة": "4",
    "خمس": "5",
    "خمسة": "5",
    "ستة": "6",
    "ست": "6",
    "سبعة": "7",
    "سبع": "7",
    "ثمانية": "8",
    "ثمان": "8",
    "تسعة": "9",
    "تسع": "9",
    "عشرة": "10",
    "عشر": "10"
}

def normalize_lesson_title(text):
    if pd.isna(text):
        return ""

    text = str(text).strip()

    # توحيد الأرقام العربية والهندية
    trans = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
    text = text.translate(trans)

    # حذف الأقواس مع إبقاء الرقم
    text = re.sub(r"[()（）]", " ", text)

    # حذف التشكيل
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)

    # توحيد بعض الحروف
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ة", "ه")
    text = text.replace("ى", "ي")

    # تحويل الأرقام المكتوبة بالحروف إلى أرقام
    for word, num in arabic_number_words.items():
        word_norm = word.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا").replace("ة", "ه")
        text = re.sub(rf"\b{word_norm}\b", num, text)

    # حذف الرموز الزائدة
    text = re.sub(r"[^\w\s]", " ", text)

    # توحيد المسافات
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
video_df["lesson_title_norm"] = video_df["lesson_title"].apply(normalize_lesson_title)
guide_df["lesson_title_norm"] = guide_df["lesson_title"].apply(normalize_lesson_title)

In [ ]:
VIDEO_LESSON_COL = "lesson_title_norm"
GUIDE_LESSON_COL = "lesson_title_norm"
VIDEO_CONCEPT_COL = "concept"
VIDEO_OBJECTIVE_COL = "learning_objective"

GUIDE_LESSON_COL = "lesson_title"
GUIDE_SKILL_COL = "guide_skill"

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

for col in video_df.columns:
    video_df[col] = video_df[col].apply(clean_text)

for col in guide_df.columns:
    guide_df[col] = guide_df[col].apply(clean_text)

In [ ]:
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
def build_video_text(row):
    return f"""
المهارة: {row[VIDEO_SKILL_COL]}
المفهوم: {row[VIDEO_CONCEPT_COL]}
الهدف التعليمي: {row[VIDEO_OBJECTIVE_COL]}
""".strip()

video_df["video_combined_text"] = video_df.apply(build_video_text, axis=1)

In [ ]:
def classify_match(score, guide_skill, video_skill):
    guide_skill = guide_skill.strip()
    video_skill = video_skill.strip()

    lexical_match = (
        guide_skill in video_skill or
        video_skill in guide_skill
    )

    if lexical_match and score >= 0.60:
        return "Covered by General Skill"

    if score >= 0.85:
        return "Exact/Semantic Match"

    elif score >= 0.70:
        return "Semantic Match"

    elif score >= 0.55:
        return "Partial Match"

    else:
        return "Missing"

In [ ]:
results = []

lessons = sorted(guide_df[GUIDE_LESSON_COL].dropna().unique())

for lesson in tqdm(lessons):

    guide_skills = guide_df[
        guide_df[GUIDE_LESSON_COL] == lesson
    ][GUIDE_SKILL_COL].dropna().unique().tolist()

    video_texts = video_df[
        video_df[VIDEO_LESSON_COL] == lesson
    ]["video_combined_text"].dropna().unique().tolist()

    if len(video_texts) == 0:
        for guide_skill in guide_skills:
            results.append({
                "lesson_title": lesson,
                "guide_skill": guide_skill,
                "best_video_skill": "",
                "similarity_score": 0,
                "match_status": "Missing",
                "reason": "No video skills found for this lesson"
            })
        continue

    guide_embeddings = model.encode(
        guide_skills,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    video_embeddings = model.encode(
        video_texts,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    similarity_matrix = util.cos_sim(
        guide_embeddings,
        video_embeddings
    )

    for i, guide_skill in enumerate(guide_skills):

        best_idx = int(np.argmax(similarity_matrix[i].cpu().numpy()))
        best_score = float(similarity_matrix[i][best_idx])
        best_video_skill = video_texts[best_idx]

        match_status = classify_match(
            best_score,
            guide_skill,
            best_video_skill
        )

        results.append({
            "lesson_title": lesson,
            "guide_skill": guide_skill,
            "best_video_skill": best_video_skill,
            "similarity_score": round(best_score, 3),
            "match_status": match_status,
            "reason": "Embedding semantic similarity comparison"
        })

100%|██████████| 51/51 [02:38<00:00,  3.10s/it]


In [ ]:
results_df = pd.DataFrame(results)

results_df.to_csv(
    "/content/embedding_skill_alignment_results.csv",
    index=False,
    encoding="utf-8-sig"
)

results_df.head()

,lesson_title,guide_skill,best_video_skill,similarity_score,match_status,reason
0,الأعداد ضمن 9999,قراءة العدد بالطُّرق المختلفة āإذا كان āأحد āأ...,,0.0,Missing,No video skills found for this lesson
1,الأعداد ضمن 9999,تمثيل العدد على لوحة المنازل,,0.0,Missing,No video skills found for this lesson
2,الأعداد ضمن 9999,تمثيل على المِعداد قد يخطئ الطا,,0.0,Missing,No video skills found for this lesson
3,الأعداد ضمن 9999,كتابة الҙأعداد ضمن - āأنْ يتعرّف الطالب مفهوم ...,,0.0,Missing,No video skills found for this lesson
4,الأعداد ضمن 9999,تمثيل العدد على المعداد,,0.0,Missing,No video skills found for this lesson


In [ ]:
matched_statuses = [
    "Exact/Semantic Match",
    "Semantic Match",
    "Covered by General Skill"
]

results_df["matched"] = results_df["match_status"].isin(matched_statuses)

overall_coverage = results_df["matched"].sum() / len(results_df)

print("Total Guide Skills:", len(results_df))
print("Matched Skills:", results_df["matched"].sum())
print("Missing/Partial Skills:", len(results_df) - results_df["matched"].sum())
print("Overall Coverage:", round(overall_coverage, 3))

Total Guide Skills: 1020
Matched Skills: 18
Missing/Partial Skills: 1002
Overall Coverage: 0.018


In [ ]:
lesson_scores = results_df.groupby("lesson_title").agg(
    total_guide_skills=("guide_skill", "count"),
    matched_skills=("matched", "sum"),
    average_similarity=("similarity_score", "mean")
).reset_index()

lesson_scores["coverage_score"] = (
    lesson_scores["matched_skills"] /
    lesson_scores["total_guide_skills"]
).round(3)

lesson_scores.to_csv(
    "/content/lesson_embedding_coverage_scores.csv",
    index=False,
    encoding="utf-8-sig"
)

lesson_scores

,lesson_title,total_guide_skills,matched_skills,average_similarity,coverage_score
0,الأعداد ضمن 9999,20,0,0.00000,0.00
1,الأعداد ضمن 99999,20,0,0.00000,0.00
2,التعرف على المتر والسنتيمتر والعلاقة بينهما,20,0,0.00000,0.00
3,التقريب,20,0,0.48975,0.00
4,الزّاوية وأنواعها,20,0,0.00000,0.00
5,الشُّعاعُ والمستقيم,20,0,0.00000,0.00
6,الضرب في العشرات,20,0,0.43785,0.00
7,الضرب في العشرات تفاعلي,20,0,0.43730,0.00
8,الضرب في المئات,20,0,0.40275,0.00
9,الضرب في المئات ومضاعفاتها,20,0,0.38580,0.00
